# 5-Fold Data Splitting

This notebook creates 5-fold cross-validation splits for insurance modeling data.

**Steps:**
1. Load and merge data
2. Remove zero exposure records
3. Calculate Pure Premium
4. Run simulations to find optimal seed
5. Apply best seed and validate
6. Save output

## 1. Setup and Imports

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import functions from utils.py
from utils import (
    load_and_merge_data,
    remove_zero_exposure,
    calculate_pure_premium,
    calculate_overall_stats,
    print_overall_stats,
    run_simulations,
    get_best_seed,
    plot_objective_distribution,
    assign_folds,
    validate_folds,
    save_output
)

# Import config
from config import (
    MAIN_DATA_PATH, SUPERPOLICY_PATH, OUTPUT_PATH,
    N_SIMULATIONS, N_FOLDS
)

print("Imports complete!")
print(f"\nConfiguration:")
print(f"  Main data: {MAIN_DATA_PATH}")
print(f"  Superpolicy: {SUPERPOLICY_PATH}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Simulations: {N_SIMULATIONS}")
print(f"  Folds: {N_FOLDS}")

## 2. Load and Merge Data

In [ ]:
# Load main data and superpolicy data, merge on vin_date
df = load_and_merge_data()

In [ ]:
# Quick look at the data
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 3. Remove Zero Exposure Records

In [ ]:
# Remove records where any exposure column is zero
df = remove_zero_exposure(df)

In [ ]:
# Verify cleaned data
print(f"Cleaned data shape: {df.shape}")
df.head()

## 4. Calculate Pure Premium

In [ ]:
# Calculate Pure Premium for each coverage
df = calculate_pure_premium(df)

In [ ]:
# Calculate overall statistics (needed for objective function)
overall_stats = calculate_overall_stats(df)
print_overall_stats(overall_stats)

## 5. Run Simulations to Find Optimal Seed

In [ ]:
# Run simulations (100 seeds by default)
# This may take a few minutes depending on data size
results_df = run_simulations(df, overall_stats)

In [ ]:
# View simulation results
results_df.head(10)

## 6. View Objective Function Distribution

In [ ]:
# Plot histogram of objective function values
plot_objective_distribution(results_df)

In [ ]:
# Find the best seed
best_seed, best_obj = get_best_seed(results_df)

## 7. Apply Best Seed and Assign Folds

In [ ]:
# Assign folds using the best seed
df = assign_folds(df, best_seed)

print(f"\nFold distribution:")
print(df['fold'].value_counts().sort_index())

## 8. Validate Fold Assignment

In [ ]:
# Validate the fold assignment quality
validate_folds(df, overall_stats)

## 9. Save Output

In [ ]:
# Save fold assignments and simulation results
df_output = save_output(df, simulation_results=results_df)

In [ ]:
# Preview final output
print(f"\nOutput preview:")
df_output.head(10)

## Done!

The fold assignments have been saved. You can now merge this output back to your original data using `vin_date` as the join key.

In [ ]:
# Example: How to merge back to original data
# original_data = pd.read_parquet("your_original_data.parquet")
# merged = pd.merge(original_data, df_output[['vin_date', 'fold']], on='vin_date', how='left')